# 03.6 — Unified Multi-Family Benchmark Aggregator

## 1. Giới thiệu & Nguyên tắc Độc lập
Notebook này **TUYỆT ĐỐI KHÔNG HUẤN LUYỆN LẠI BẤT KỲ MÔ HÌNH NÀO**.

### Nhiệm vụ Cốt lõi:
1. Đọc và hợp nhất 5 file artifact kết quả từ các notebook chuyên biệt:
   - `data/processed/results/statistical_results.csv`
   - `data/processed/results/ml_direct_results.csv`
   - `data/processed/results/ml_hurdle_results.csv`
   - `data/processed/results/ml_tweedie_results.csv`
   - `data/processed/results/neural_results.csv`
2. Kiểm tra tính toàn vẹn của Schema dữ liệu thống nhất (`UNIFIED_SCHEMA`).
3. Xây dựng Bảng Xếp hạng Tổng thể (**Global Leaderboard**).
4. Xác định Champion của từng nhóm mô hình và Overall Champion.
5. Xuất file kết quả tổng thể chuẩn hóa: `data/processed/results/unified_benchmark_results.csv`.


In [1]:
import sys
from pathlib import Path
import json
import time
import warnings
warnings.filterwarnings('ignore')

# Tự động tìm project root từ bất kỳ thư mục con nào
ROOT_DIR = Path.cwd()
while not (ROOT_DIR / "src").exists() and ROOT_DIR != ROOT_DIR.parent:
    ROOT_DIR = ROOT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.training.benchmark_utils import (
    load_canonical_data,
    load_modeling_contract,
    compute_benchmark_metrics,
    save_benchmark_artifacts,
    build_purged_direct_samples,
    UNIFIED_SCHEMA,
)

results_dir = ROOT_DIR / "data" / "processed" / "results"
expected_files = [
    "statistical_results.csv",
    "ml_direct_results.csv",
    "ml_hurdle_results.csv",
    "ml_tweedie_results.csv",
    "neural_results.csv"
]

print("Checking presence of result artifacts...")
all_exist = True
for f in expected_files:
    p = results_dir / f
    status = "EXISTS" if p.exists() else "MISSING"
    print(f" - {f:30s}: {status}")
    if not p.exists():
        all_exist = False

assert all_exist, "Cannot aggregate: Some group results are missing! Run upstream modeling notebooks first."
print("\n>> All 5 group artifacts are present and ready for aggregation.")


Checking presence of result artifacts...
 - statistical_results.csv       : EXISTS
 - ml_direct_results.csv         : EXISTS
 - ml_hurdle_results.csv         : EXISTS
 - ml_tweedie_results.csv        : EXISTS
 - neural_results.csv            : EXISTS

>> All 5 group artifacts are present and ready for aggregation.


## 2. Nạp & Hợp nhất Toàn bộ Kết quả vào DataFrame Duy nhất

In [2]:
dfs = []
for f in expected_files:
    df_part = pd.read_csv(results_dir / f)
    assert list(df_part.columns) == UNIFIED_SCHEMA, f"Schema mismatch in {f}!"
    dfs.append(df_part)

unified_df = pd.concat(dfs, ignore_index=True)
print(f"Total benchmarked models across all families: {len(unified_df)}")


Total benchmarked models across all families: 26


## 3. Bảng Xếp hạng Toàn cầu (Global Leaderboard by CR-MAE@7)

In [3]:
unified_df = unified_df.sort_values(by='CR-MAE@7').reset_index(drop=True)
unified_df.index = unified_df.index + 1

unified_csv_path = results_dir / "unified_benchmark_results.csv"
unified_df.to_csv(unified_csv_path, index=False)
print(f">> Saved global unified benchmark to: {unified_csv_path}\n")

print("=== GLOBAL BENCHMARK LEADERBOARD (Top 15 Models) ===")
display(unified_df[['model', 'group', 'strategy', 'MAE', 'RMSE', 'rain_day_MAE', 'CR-MAE@7', 'CR-Bias@7', 'training_time']].head(15))


>> Saved global unified benchmark to: D:\DS108_project\data\processed\results\unified_benchmark_results.csv

=== GLOBAL BENCHMARK LEADERBOARD (Top 15 Models) ===
                  model                 group  ... CR-Bias@7  training_time
1      LightGBM_Tweedie  Tweedie Single-Stage  ...   -9.1059          0.839
2       XGBoost_Tweedie  Tweedie Single-Stage  ...   -8.9689          1.743
3      LinearRegression      Machine Learning  ...   -7.1560          0.117
4       LightGBM_Direct      Machine Learning  ...   -4.5106          0.538
5          RandomForest      Machine Learning  ...   -5.5001          1.273
6        XGBoost_Direct      Machine Learning  ...   -2.5995          1.513
7        XGBoost_Hurdle      Hurdle Two-Stage  ...  -17.3513          0.666
8       LightGBM_Hurdle      Hurdle Two-Stage  ...  -19.8137          0.276
9   RandomForest_Hurdle      Hurdle Two-Stage  ...  -18.3404          9.142
10      Logistic+Linear      Hurdle Two-Stage  ...  -18.1241          0.102
11

## 4. Xác định Champion Từng Nhóm & Overall Champion

In [4]:
champions = []
for group_name, grp in unified_df.groupby('group'):
    champ = grp.sort_values(by='CR-MAE@7').iloc[0]
    champions.append(champ)

df_champions = pd.DataFrame(champions).sort_values(by='CR-MAE@7').reset_index(drop=True)
df_champions.index = df_champions.index + 1

print("=== GROUP CHAMPIONS COMPARISON ===")
display(df_champions[['group', 'model', 'strategy', 'MAE', 'RMSE', 'rain_day_MAE', 'CR-MAE@7', 'CR-Bias@7', 'training_time', 'parameter_count']])

overall_champ = df_champions.iloc[0]
print(f"\n🏆 OVERALL PROJECT CHAMPION: {overall_champ['model']} ({overall_champ['group']}) with CR-MAE@7 = {overall_champ['CR-MAE@7']:.4f} mm")


=== GROUP CHAMPIONS COMPARISON ===
                  group             model  ... training_time  parameter_count
1  Tweedie Single-Stage  LightGBM_Tweedie  ...         0.839              600
2      Machine Learning  LinearRegression  ...         0.117              100
3      Hurdle Two-Stage    XGBoost_Hurdle  ...         0.666              800
4           Statistical             ARIMA  ...         1.817                5
5              Baseline     SeasonalNaive  ...         1.817                2
6     Neural Forecaster          PatchTST  ...         2.652            30000

[6 rows x 10 columns]

🏆 OVERALL PROJECT CHAMPION: LightGBM_Tweedie (Tweedie Single-Stage) with CR-MAE@7 = 17.3842 mm
